# 09c Weather-Tail Post-Estimation Analysis

This notebook evaluates whether the M2 weather signal differs on unusual realised-weather days. It reuses the row-level rolling-origin predictions from notebook 09, joins realised `trade_08_19` weather only after predictions are fixed, and compares M1 and M2 forecast errors within weather-tail groups.

The analysis is post-estimation. It does not train, tune, rerun notebook 09, or regenerate predictions. Realised weather is used only for diagnostic grouping, not as a model input.


## Setup and paths

This cell resolves project paths and defines the dedicated output folder for the weather-tail analysis.


In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

MARKER_FILES = ("README_FOR_CODEX.md", "AGENTS.md", "pyproject.toml")


def find_project_root(start=None):
    cur = Path.cwd() if start is None else Path(start).resolve()
    for c in [cur, *cur.parents]:
        if any((c / m).exists() for m in MARKER_FILES):
            return c
    raise FileNotFoundError("project root not found")


PROJECT_ROOT = find_project_root()
PROCESSED = PROJECT_ROOT / "data" / "processed"
RO_DIR = PROJECT_ROOT / "outputs" / "lightgbm_rolling_origin"
PRED_PATH = RO_DIR / "lightgbm_rolling_origin_predictions_full_step5.parquet"
REALISED_PATH = PROCESSED / "realised_weather_daily_windows.parquet"

# Dedicated output folder for this post-estimation analysis.
OUT_DIR = RO_DIR / "full_step5_weather_tail_analysis"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = "trade_08_19"  # operational weather window used by the emulator/model
MIN_CELL_OBS = 12  # minimum store-days per percentile cell before falling back

print("Project root :", PROJECT_ROOT)
print("Predictions  :", PRED_PATH)
print("Realised wx  :", REALISED_PATH)
print("Output dir   :", OUT_DIR)


## Load schemas and inputs

This cell loads only the prediction and realised-weather columns needed for the analysis. Predictions are restricted to M1 and M2, and realised weather is restricted to the operational `trade_08_19` window before joining.


In [ ]:
WEATHER_VARS = {
    "temp": "temp_obs_mean",
    "precip": "precip_obs_sum",
    "wind": "wind_obs_mean",
    "humid": "humid_obs_mean",
    "cloud": "cloud_obs_mean",
}

PRED_COLS = [
    "AvdelingID",
    "Analyse_Kategori",
    "origin_date",
    "target_date",
    "horizon",
    "feature_set",
    "y_true",
    "y_pred",
]
pred = pd.read_parquet(PRED_PATH, columns=PRED_COLS)
pred = pred[pred["feature_set"].isin(["M1", "M2"])].copy()
pred["target_date"] = pd.to_datetime(pred["target_date"]).dt.normalize()
pred["AvdelingID"] = pd.to_numeric(pred["AvdelingID"], errors="coerce").astype("int64")

WX_COLS = ["date", "AvdelingID", "aggregation_window"] + list(WEATHER_VARS.values())
wx = pd.read_parquet(REALISED_PATH, columns=WX_COLS)
wx = wx[wx["aggregation_window"].astype("string") == WINDOW].copy()
wx["date"] = pd.to_datetime(wx["date"]).dt.normalize()
wx["AvdelingID"] = pd.to_numeric(wx["AvdelingID"], errors="coerce").astype("int64")
wx = wx.rename(columns={"date": "target_date"})

print(f"Predictions (M1+M2) rows : {len(pred):,}")
print(f"Realised wx ({WINDOW}) rows : {len(wx):,}")
print("Realised weather columns mapped:", WEATHER_VARS)


## Input validation

This cell checks that required inputs and columns are present, that realised-weather keys are unique after filtering, that both M1 and M2 predictions are available, and that prediction-to-weather join coverage is sufficient.


In [ ]:
assert PRED_PATH.exists(), f"missing predictions: {PRED_PATH}"
assert REALISED_PATH.exists(), f"missing realised weather: {REALISED_PATH}"
for c in PRED_COLS:
    assert c in pred.columns, f"prediction missing column: {c}"
for v in WEATHER_VARS.values():
    assert v in wx.columns, f"realised weather missing column: {v}"

dup = int(wx.duplicated(["target_date", "AvdelingID"]).sum())
assert dup == 0, f"realised weather has {dup} duplicate (date, AvdelingID) keys after trade_08_19 filter"

fs = set(pred["feature_set"].unique())
assert {"M1", "M2"}.issubset(fs), f"need both M1 and M2; found {fs}"

pk = pred[["target_date", "AvdelingID"]].drop_duplicates()
cov = pk.merge(
    wx[["target_date", "AvdelingID"]].assign(_hit=1),
    on=["target_date", "AvdelingID"],
    how="left",
)
coverage = float(cov["_hit"].notna().mean())
print(
    f"Prediction store-days: {len(pk):,} | matched in realised weather: "
    f"{int(cov['_hit'].notna().sum()):,} ({coverage:.3%})"
)
assert coverage > 0.99, f"join coverage too low: {coverage:.3%}"
print("Validation passed.")


## Error construction and weather join

Forecast errors are computed from `y_true` and `y_pred`. Realised weather is joined as a many-predictions-to-one-weather-row merge, so the join does not duplicate prediction rows.

The join is leakage-safe for this diagnostic purpose because model predictions are already fixed. Realised target-day weather is used only to partition completed predictions into post-estimation subgroups.


In [ ]:
joined = pred.merge(
    wx[["target_date", "AvdelingID"] + list(WEATHER_VARS.values())],
    on=["target_date", "AvdelingID"],
    how="inner",
    validate="m:1",
)
err = joined["y_pred"].astype("float64") - joined["y_true"].astype("float64")
joined["error"] = err
joined["abs_error"] = err.abs()
joined["squared_error"] = err ** 2
joined["month"] = joined["target_date"].dt.month.astype("int8")
SEASON_MAP = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn",
}
joined["season"] = joined["month"].map(SEASON_MAP).astype("string")
print(f"Joined rows: {len(joined):,} | prediction rows not matched: {len(pred) - len(joined):,}")


## Weather-tail construction

Weather-tail thresholds are estimated from realised `trade_08_19` weather in the evaluation store-days. The basis is selected from the most local available grouping with at least `MIN_CELL_OBS` observations, falling back from `local_month` to `store_season` and then to `month_across_stores` when needed.

Tail groups are overlapping: a store-day may enter several groups. The groups are `all_days`, `cold_tail`, `warm_tail`, `wet_tail`, `windy_tail`, `cloudy_tail` and `clear_tail`.


In [ ]:
day_wx = (
    joined[["target_date", "AvdelingID", "month", "season"] + list(WEATHER_VARS.values())]
    .drop_duplicates(["target_date", "AvdelingID"])
    .copy()
)


def min_cell(basis_cols):
    return int(day_wx.groupby(basis_cols, observed=True).size().min())


CANDIDATE_BASES = [
    ("local_month", ["AvdelingID", "month"]),
    ("store_season", ["AvdelingID", "season"]),
    ("month_across_stores", ["month"]),
]
TAIL_BASIS, BASIS_COLS = None, None
for name, cols in CANDIDATE_BASES:
    if min_cell(cols) >= MIN_CELL_OBS:
        TAIL_BASIS, BASIS_COLS = name, cols
        break
if TAIL_BASIS is None:
    TAIL_BASIS, BASIS_COLS = "month_across_stores", ["month"]
print(
    f"Tail percentile basis: {TAIL_BASIS} "
    f"(cols={BASIS_COLS}; min cell obs={min_cell(BASIS_COLS)})"
)

g = day_wx.groupby(BASIS_COLS, observed=True)
for var, col in WEATHER_VARS.items():
    day_wx[f"{var}_p10"] = g[col].transform(lambda s: s.quantile(0.10))
    day_wx[f"{var}_p90"] = g[col].transform(lambda s: s.quantile(0.90))

THR_COLS = [c for c in day_wx.columns if c.endswith("_p10") or c.endswith("_p90")]
joined = joined.merge(
    day_wx[["target_date", "AvdelingID"] + THR_COLS],
    on=["target_date", "AvdelingID"],
    how="left",
)

V = WEATHER_VARS
masks = {
    "all_days": pd.Series(True, index=joined.index),
    "cold_tail": joined[V["temp"]] <= joined["temp_p10"],
    "warm_tail": joined[V["temp"]] >= joined["temp_p90"],
    "wet_tail": joined[V["precip"]] >= joined["precip_p90"],
    "windy_tail": joined[V["wind"]] >= joined["wind_p90"],
    "cloudy_tail": joined[V["cloud"]] >= joined["cloud_p90"],
    "clear_tail": joined[V["cloud"]] <= joined["cloud_p10"],
}
TAIL_ORDER = [
    "all_days",
    "cold_tail",
    "warm_tail",
    "wet_tail",
    "windy_tail",
    "cloudy_tail",
    "clear_tail",
]

m1 = joined[joined["feature_set"] == "M1"]
print("Store-day counts per tail group:")
for k in TAIL_ORDER:
    sd = m1[masks[k].loc[m1.index]][["target_date", "AvdelingID"]].drop_duplicates()
    print(f"  {k:11s}: {len(sd):,} store-days")


## Metric functions

This cell computes `n`, RMSE, MAE and WAPE by tail group, optional category, horizon and feature set. M1-vs-M2 improvements are reported so that positive values indicate lower error for M2.


In [ ]:
def _metrics(d):
    yt = d["y_true"].to_numpy("float64")
    e = d["error"].to_numpy("float64")
    syt = float(np.sum(yt))
    n = len(d)
    return pd.Series(
        {
            "n": n,
            "rmse": math.sqrt(np.mean(e ** 2)) if n else np.nan,
            "mae": float(np.mean(np.abs(e))) if n else np.nan,
            "wape": float(np.sum(np.abs(e)) / syt) if syt else np.nan,
        }
    )


def compute(group_cols, row_mask=None):
    """Stack overlapping tail subsets into a tagged long frame, then metrics + improvements."""
    frames = []
    for k in TAIL_ORDER:
        mk = masks[k] if row_mask is None else (masks[k] & row_mask)
        sub = joined[mk].copy()
        sub["tail_group"] = k
        frames.append(sub)
    tg = pd.concat(frames, ignore_index=True)
    keys = ["tail_group"] + group_cols + ["feature_set"]
    long = tg.groupby(keys, observed=True)[["y_true", "error"]].apply(_metrics).reset_index()
    wide = long.pivot_table(
        index=["tail_group"] + group_cols,
        columns="feature_set",
        values=["n", "rmse", "mae", "wape"],
    )
    wide.columns = [f"{a}_{b}" for a, b in wide.columns]
    wide = wide.reset_index()
    wide["RMSE_improvement_abs"] = wide["rmse_M1"] - wide["rmse_M2"]
    wide["RMSE_improvement_pct"] = 100.0 * (wide["rmse_M1"] - wide["rmse_M2"]) / wide["rmse_M1"]
    wide["MAE_improvement_pct"] = 100.0 * (wide["mae_M1"] - wide["mae_M2"]) / wide["mae_M1"]
    wide["WAPE_improvement_pct"] = 100.0 * (wide["wape_M1"] - wide["wape_M2"]) / wide["wape_M1"]
    wide["tail_group"] = pd.Categorical(wide["tail_group"], TAIL_ORDER, ordered=True)
    return wide.sort_values(["tail_group"] + group_cols).reset_index(drop=True)


H3_H10 = joined["horizon"].between(3, 10)
by_tail_horizon = compute(["horizon"])
by_tail_cat_horizon = compute(["Analyse_Kategori", "horizon"])
summary_h3_h10 = compute([], row_mask=H3_H10)
summary_h3_h10_by_cat = compute(["Analyse_Kategori"], row_mask=H3_H10)

print("by_tail_horizon rows:", len(by_tail_horizon))
print("\nPooled h=3-10 summary (M2 vs M1):")
print(summary_h3_h10[["tail_group", "n_M1", "rmse_M1", "rmse_M2", "RMSE_improvement_pct"]].to_string(index=False))


## Output tables

Detailed output tables retain all available horizons. Compact thesis-facing summaries focus on the h=3..10 medium-range window.


In [ ]:
P1 = OUT_DIR / "weather_tail_m1_m2_metrics_by_tail_horizon.csv"
P2 = OUT_DIR / "weather_tail_m1_m2_metrics_by_tail_category_horizon.csv"
P3 = OUT_DIR / "weather_tail_m1_m2_summary_h3_h10.csv"
P4 = OUT_DIR / "weather_tail_m1_m2_summary_h3_h10_by_category.csv"

by_tail_horizon.to_csv(P1, index=False, encoding="utf-8-sig")
by_tail_cat_horizon.to_csv(P2, index=False, encoding="utf-8-sig")
summary_h3_h10.to_csv(P3, index=False, encoding="utf-8-sig")
summary_h3_h10_by_cat.to_csv(P4, index=False, encoding="utf-8-sig")
for p in (P1, P2, P3, P4):
    print("saved:", p.relative_to(PROJECT_ROOT).as_posix())


## Figures

This optional section writes thesis-style figures for h=3..10 weather-tail improvements. Positive values indicate that M2 improves on M1.


In [ ]:
# Weather-tail RMSE improvement for pooled h=3..10.
d = summary_h3_h10.copy()
d["tail_group"] = d["tail_group"].astype(str)
d = (
    d.set_index("tail_group")
    .loc[[t for t in TAIL_ORDER if t in d["tail_group"].values]]
    .reset_index()
)
vals = d["RMSE_improvement_pct"].to_numpy()
fig, ax = plt.subplots(figsize=(8.6, 4.8))
ax.bar(d["tail_group"], vals, color=np.where(vals >= 0, "#2E7D6B", "#A43E3E"))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("M2 vs M1 RMSE improvement by weather tail (h=3-10)")
ax.set_ylabel("RMSE improvement vs M1 (%)")
ax.set_xlabel("Weather tail group")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=30, ha="right")
for i, v in enumerate(vals):
    if np.isfinite(v):
        ax.text(
            i,
            v,
            f"{v:.1f}",
            ha="center",
            va="bottom" if v >= 0 else "top",
            fontsize=8,
        )
fig.tight_layout()
F1 = FIG_DIR / "weather_tail_rmse_improvement_pct_h3_h10.png"
fig.savefig(F1, dpi=300, bbox_inches="tight")
plt.close(fig)

# Category-by-tail RMSE improvement heatmap for pooled h=3..10.
piv = summary_h3_h10_by_cat.copy()
piv["tail_group"] = piv["tail_group"].astype(str)
P = piv.pivot(index="Analyse_Kategori", columns="tail_group", values="RMSE_improvement_pct")
P = P[[t for t in TAIL_ORDER if t in P.columns]]
if "all_days" in P.columns:
    P = P.loc[P["all_days"].sort_values(ascending=False).index]
vmax = float(np.nanmax(np.abs(P.values)))
fig, ax = plt.subplots(figsize=(9.6, 4.8))
im = ax.imshow(
    P.values,
    aspect="auto",
    cmap="RdYlGn",
    norm=TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax),
)
ax.set_xticks(range(len(P.columns)))
ax.set_xticklabels(P.columns, rotation=30, ha="right")
ax.set_yticks(range(len(P.index)))
ax.set_yticklabels(P.index)
ax.set_title("M2 vs M1 RMSE improvement (%) by category and weather tail (h=3-10)")
ax.set_xlabel("Weather tail group")
ax.set_ylabel("Product category")
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        v = P.iloc[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=7)
cb = fig.colorbar(im, ax=ax)
cb.set_label("RMSE improvement vs M1 (%)")
fig.tight_layout()
F2 = FIG_DIR / "weather_tail_rmse_improvement_pct_by_category_h3_h10.png"
fig.savefig(F2, dpi=300, bbox_inches="tight")
plt.close(fig)
print("saved:", F1.relative_to(PROJECT_ROOT).as_posix())
print("saved:", F2.relative_to(PROJECT_ROOT).as_posix())


## Interpretation notes

Weather-tail results indicate whether the average M2 improvement is concentrated on unusual realised-weather days or spread more evenly across all days. Negative or weak tail improvements should be interpreted cautiously because tail groups can be small and may also capture holidays, closures, campaigns or other non-weather conditions.

For h=3..10, M2 uses the synthetic medium-range emulator. Results in this window are therefore scenario evidence conditional on the emulator, not validation against archived real medium-range operational forecasts.


## Final summary

This cell prints the main output locations and completion status for the post-estimation weather-tail analysis.


In [ ]:
print("=" * 64)
print("09c WEATHER-TAIL POST-ESTIMATION SUMMARY")
print("=" * 64)
print(f"Input prediction rows (M1+M2): {len(pred):,}")
print(f"Joined rows                  : {len(joined):,}")
print(f"Join coverage (store-days)   : {coverage:.3%}")
print(f"Tail percentile basis        : {TAIL_BASIS} {BASIS_COLS}")
print("Tail store-day counts:")
for k in TAIL_ORDER:
    sd = m1[masks[k].loc[m1.index]][["target_date", "AvdelingID"]].drop_duplicates()
    print(f"  {k:11s}: {len(sd):,}")
print("Output tables:")
for p in (P1, P2, P3, P4):
    print("  ", p.relative_to(PROJECT_ROOT).as_posix())
print("Figures:")
for f in (F1, F2):
    print("  ", f.relative_to(PROJECT_ROOT).as_posix())
print(
    "No ML was run; no predictions were regenerated. "
    "Diagnostic post-estimation analysis only."
)
